# ANTARES Cumulative Nightly History

This notebook builds the Drive-backed nightly history store without replacing the normal `alerts_time_comparison.ipynb` workflow.

In [ ]:
%cd /content
!rm -rf ANTARES_Analysis
!git clone https://github.com/darim1151/ANTARES_Analysis.git
%cd ANTARES_Analysis
!git pull --ff-only
!ls src

In [ ]:
!pip install --quiet antares-client elasticsearch-dsl astropy matplotlib pandas numpy pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/ANTARES_Analysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, history

DATA_ROOT = config.HISTORY_DATA_ROOT
MJD_HISTORY_START = config.MJD2_MIN
MJD_HISTORY_CUTOFF = config.MJD2_MAX

print('Ready.')
config.print_config_summary()
print('\nHistory store:', DATA_ROOT)

## 1. Tiny Smoke Backfill

This creates one nightly partition with a small target and skips lightcurves. Use it first to verify Drive paths, manifests, parquet writing, and resume behavior.

In [ ]:
smoke_summary = history.backfill_history(
    data_root=DATA_ROOT,
    mjd_start=MJD_HISTORY_START,
    mjd_stop=MJD_HISTORY_START + 1,
    target_loci=1000,
    max_nights=1,
    fetch_lightcurves=False,
    resume=True,
)
display(smoke_summary.tail())

In [ ]:
resume_summary = history.backfill_history(
    data_root=DATA_ROOT,
    mjd_start=MJD_HISTORY_START,
    mjd_stop=MJD_HISTORY_START + 1,
    target_loci=1000,
    max_nights=1,
    fetch_lightcurves=False,
    resume=True,
)
display(resume_summary.tail())

## 2. Three-Night Test

Turn this on after the smoke test. It checks multi-night folder layout and cumulative-index behavior.

In [ ]:
RUN_THREE_NIGHT_TEST = False

if RUN_THREE_NIGHT_TEST:
    three_night_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_START + 3,
        target_loci=5000,
        max_nights=3,
        fetch_lightcurves=False,
        resume=True,
    )
    display(three_night_summary.tail(10))

## 3. Full-Scale Runs

Run one full night first. Only then turn on the full historical backfill.

In [ ]:
RUN_ONE_FULL_NIGHT = False
RUN_FULL_BACKFILL = False

if RUN_ONE_FULL_NIGHT:
    one_full_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_START + 1,
        target_loci=config.HISTORY_TARGET_LOCI,
        max_nights=1,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    display(one_full_summary.tail())

if RUN_FULL_BACKFILL:
    full_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_CUTOFF,
        target_loci=config.HISTORY_TARGET_LOCI,
        max_nights=None,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    display(full_summary.tail(20))

## 4. Nightly Update

This ingests the newest night, compares it against prior cumulative history, and appends it only after validation passes.

In [ ]:
RUN_NIGHTLY_UPDATE = False

if RUN_NIGHTLY_UPDATE:
    nightly_result = history.run_nightly_update(
        data_root=DATA_ROOT,
        mjd_min=config.MJD1_MIN,
        mjd_max=config.MJD1_MAX,
        target_loci=config.HISTORY_TARGET_LOCI,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    print(nightly_result['comparison'])
    print(nightly_result['manifest']['status'])

## 5. Inspect Stored Data

In [ ]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)
print(f'Cumulative index rows: {len(loci_index):,}')
display(nightly_summary.tail(20))